# Chapter 3 - Lab 4: AutoGen agent

**Goal**: build the agent with **AutoGen AgentChat**, where agents coordinate through conversation.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **AutoGen** (`autogen-agentchat`, `autogen-ext[openai]`) - `AssistantAgent`, `RoundRobinGroupChat`.
* **OpenAI** `gpt-4o-mini`.

## 1. Install packages

In [ ]:
%pip install -q autogen-agentchat 'autogen-ext[openai]' pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Model client, tools and agent

In [ ]:
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_core.tools import FunctionTool
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])

def support_data(product_line: str) -> str:
    """Get the quarter's support tickets and delivered orders for a product line."""
    d = get_support_data(product_line)
    return f"{product_line}: tickets={d.tickets} orders={d.orders}"

def contact_rate(tickets: int, orders: int) -> float:
    """Compute the contact rate: support tickets per 100 orders."""
    return compute_contact_rate(tickets, orders)

analyst = AssistantAgent(
    name="CXAnalyst",
    model_client=model_client,
    tools=[FunctionTool(support_data, description="Tickets and orders for a product line"),
           FunctionTool(contact_rate, description="Tickets per 100 orders")],
    system_message=system_message + " When the memo is complete, end with the word TERMINATE.",
    reflect_on_tool_use=True,
)

## 5. Run as a (one-member) team with a termination rule

Conversational frameworks need an explicit stop rule. We stop when the agent writes `TERMINATE`, or after 10 messages.

In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    participants=[analyst],
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(10),
)
result = await team.run(task=input_message)
for msg in result.messages:
    print(f"[{msg.source}] {getattr(msg, 'content', '')}\n")
await model_client.close()

## 6. Results
**What to notice about AutoGen**: coordination happens through the message history. Powerful for
open-ended analysis (Chapter 7, Lab 3 lets an AutoGen agent write and run its own pandas code on
ticket data), but you must design termination carefully.